# Area Classifier — Keyword Features (Colab)

Esperimento: embedding già computati + feature booleane keyword → migliora il Macro F1?

**File da caricare (cella 2):**
- `dataset_clean.csv`
- `area_v2_e5_train.npy`
- `area_v2_e5_test.npy`
- `area_v2_labels_train.npy`
- `area_v2_labels_test.npy`

In [ ]:
!pip install -q scikit-learn pandas numpy scipy

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import warnings
import scipy.sparse as sp
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings('ignore')
print('Import OK')

---
## STEP 1 — Carica i file

Carica nell'ordine: `dataset_clean.csv`, poi i 4 file `.npy`

In [ ]:
from google.colab import files

uploaded = files.upload()  # seleziona tutti e 5 i file insieme
print('\nFile caricati:', list(uploaded.keys()))

---
## STEP 2 — Caricamento embedding, label e CSV

In [ ]:
SOGLIA_SPLIT = '2025-11-01'

# Embedding e label
X_train = np.load('area_v2_e5_train.npy')
X_test  = np.load('area_v2_e5_test.npy')
y_train = np.load('area_v2_labels_train.npy', allow_pickle=True)
y_test  = np.load('area_v2_labels_test.npy',  allow_pickle=True)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')

# CSV — stesso preprocessing di area_v2
df = pd.read_csv('dataset_clean.csv', parse_dates=['data_creazione'])
df['area_v2'] = df['area'].replace({'Hardware': 'area_sistemistica'})
CLASSI_DROP = ['business_intelligence', 'protocollo_delibere']
df.loc[df['area_v2'].isin(CLASSI_DROP), 'area_v2'] = np.nan
conteggi = df['area_v2'].value_counts()
classi_rumore = conteggi[conteggi < 10].index.tolist()
if classi_rumore:
    df.loc[df['area_v2'].isin(classi_rumore), 'area_v2'] = np.nan

df_validi    = df.dropna(subset=['area_v2'])
df_train_doc = df_validi[df_validi['data_creazione'] < SOGLIA_SPLIT].reset_index(drop=True)
df_test_doc  = df_validi[df_validi['data_creazione'] >= SOGLIA_SPLIT].reset_index(drop=True)

print(f'\nTrain CSV: {len(df_train_doc):,}   y_train: {len(y_train):,}')
print(f'Test  CSV: {len(df_test_doc):,}    y_test:  {len(y_test):,}')

assert len(df_train_doc) == len(y_train), 'MISMATCH TRAIN'
assert len(df_test_doc)  == len(y_test),  'MISMATCH TEST'
print('\nAllineamento OK ✓')

---
## STEP 3 — Definizione keyword

Pattern regex per le coppie più confuse dall'analisi errori.

In [ ]:
KEYWORD_FEATURES = {
    # Ciclo Passivo
    'kw_fornitore':        r'\bfornitor[ei]\b',
    'kw_fattura_ricevuta': r'\bfattur[ae]\s+(ricevut[ae]|passiv[ae])\b',
    'kw_acquisto':         r'\bacquist[oi]\b',
    'kw_ordine_acquisto':  r'\bordine\s+(di\s+)?acquisto\b',
    'kw_registraz_fatt':   r'\bregistrazione\s+fattur[ae]\b',
    'kw_nota_credito':     r'\bnota\s+credito\b',

    # Ciclo Attivo
    'kw_ospite':           r'\bospite\b',
    'kw_parente':          r'\bparent[ei]\b',
    'kw_retta':            r'\brette?\b',
    'kw_fattura_emessa':   r'\bfattur[ae]\s+(emess[ae]|attiv[ae])\b',
    'kw_emissione':        r'\bemission[ei]\b',
    'kw_cartella_utente':  r'\bcartella\s+utente\b',
    'kw_cliente':          r'\bclient[ei]\b',

    # Area Sanitaria
    'kw_sipcar':           r'\bsipcar\b',
    'kw_paziente':         r'\bpazient[ei]\b',
    'kw_cartella_clinica': r'\bcartella\s+(clinica|sanitaria|del\s+paziente)\b',
    'kw_ricovero':         r'\bricovero\b',
    'kw_rsa':              r'\bRSA\b',
    'kw_prescrizione':     r'\bprescrizioni?\b',
    'kw_diagnosi':         r'\bdiagnosi\b',
    'kw_degenza':          r'\bdegenza\b',
    'kw_farmacia':         r'\bfarmaci[ae]?\b',

    # Area Sistemistica
    'kw_server':           r'\bserver\b',
    'kw_vpn':              r'\bvpn\b',
    'kw_firewall':         r'\bfirewall\b',
    'kw_backup':           r'\bbackup\b',
    'kw_database':         r'\bdatabase\b',
    'kw_rete_infra':       r'\binfrastruttur[ae]\b',
    'kw_windows':          r'\bwindows\b',
    'kw_sql':              r'\b(sql|mysql|postgresql)\b',

    # Area Personale
    'kw_paghe':            r'\bpaghe?\b',
    'kw_stipendio':        r'\bstipendi[oi]\b',
    'kw_cedolino':         r'\bcedolin[oi]\b',
    'kw_presenze':         r'\bpresenze\b',
    'kw_timbrature':       r'\btimbratur[ae]\b',
    'kw_bilancio_prev':    r'\bbilancio\s+(di\s+)?previsione\b',
    'kw_tfr':              r'\btfr\b',
    'kw_ferie':            r'\bferie\b',
    'kw_assunzione':       r'\bassunzione\b',
    'kw_contratto_lavoro': r'\bcontratto\s+(di\s+)?lavoro\b',
}

print(f'Keyword definite: {len(KEYWORD_FEATURES)}')

---
## STEP 4 — Costruzione matrice keyword (booleana)

In [ ]:
def build_keyword_matrix(testi, keyword_dict):
    """Restituisce matrice booleana sparsa (n_tickets × n_keywords)."""
    patterns = {nome: re.compile(pat, re.IGNORECASE) for nome, pat in keyword_dict.items()}
    nomi = list(keyword_dict.keys())
    righe = []
    for testo in testi:
        s = str(testo) if pd.notna(testo) else ''
        righe.append([1 if p.search(s) else 0 for p in patterns.values()])
    return sp.csr_matrix(np.array(righe, dtype=np.float32)), nomi


print('Building keyword matrix...')
t0 = time.time()
KW_train, nomi_kw = build_keyword_matrix(df_train_doc['testo_input'], KEYWORD_FEATURES)
KW_test,  _       = build_keyword_matrix(df_test_doc['testo_input'],  KEYWORD_FEATURES)
print(f'Fatto in {time.time()-t0:.1f}s')
print(f'KW_train: {KW_train.shape}  |  KW_test: {KW_test.shape}')

# Copertura: quanti ticket hanno almeno 1 keyword?
hit_tr = (KW_train.sum(axis=1) > 0).sum()
hit_te = (KW_test.sum(axis=1)  > 0).sum()
print(f'\nTicket con ≥1 keyword — train: {hit_tr:,}/{KW_train.shape[0]:,} ({hit_tr/KW_train.shape[0]*100:.1f}%)')
print(f'Ticket con ≥1 keyword — test:  {hit_te:,}/{KW_test.shape[0]:,}  ({hit_te/KW_test.shape[0]*100:.1f}%)')

---
## STEP 5 — hstack: embedding + keyword

In [ ]:
# hstack: unisce colonna per colonna i due blocchi di feature
X_train_combined = sp.hstack([sp.csr_matrix(X_train), KW_train])
X_test_combined  = sp.hstack([sp.csr_matrix(X_test),  KW_test])

print(f'Baseline  — train: {X_train.shape}         test: {X_test.shape}')
print(f'Combinato — train: {X_train_combined.shape}  test: {X_test_combined.shape}')
print(f'  (768 embedding + {len(nomi_kw)} keyword = {768 + len(nomi_kw)} feature totali)')

---
## STEP 6 — Confronto: Baseline vs Embedding + Keyword

In [ ]:
C = 10.0  # ottimale da sessione VSCode

# ── Baseline ──────────────────────────────────────────────────────────────
print('=== BASELINE — Embedding solo ===')
t0 = time.time()
clf_base = LinearSVC(class_weight='balanced', C=C, max_iter=3000, random_state=42)
clf_base.fit(X_train, y_train)
y_pred_base = clf_base.predict(X_test)
f1_base = f1_score(y_test, y_pred_base, average='macro')
print(f'Macro F1: {f1_base:.4f}  |  Accuracy: {(y_pred_base == y_test).mean():.4f}  ({time.time()-t0:.1f}s)')
print()

# ── Combinato ─────────────────────────────────────────────────────────────
print('=== COMBINATO — Embedding + Keyword ===')
t0 = time.time()
clf_comb = LinearSVC(class_weight='balanced', C=C, max_iter=3000, random_state=42)
clf_comb.fit(X_train_combined, y_train)
y_pred_comb = clf_comb.predict(X_test_combined)
f1_comb = f1_score(y_test, y_pred_comb, average='macro')
print(f'Macro F1: {f1_comb:.4f}  |  Accuracy: {(y_pred_comb == y_test).mean():.4f}  ({time.time()-t0:.1f}s)')

print()
print(f'Delta Macro F1: {f1_comb - f1_base:+.4f}')

---
## STEP 7 — Delta F1 per classe

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

classi = sorted(clf_base.classes_)
_, _, f1_b, _ = precision_recall_fscore_support(y_test, y_pred_base, labels=classi, zero_division=0)
_, _, f1_c, _ = precision_recall_fscore_support(y_test, y_pred_comb, labels=classi, zero_division=0)

df_delta = pd.DataFrame({'classe': classi, 'f1_base': f1_b.round(4), 'f1_comb': f1_c.round(4)})
df_delta['delta'] = (f1_c - f1_b).round(4)
df_delta['esito'] = df_delta['delta'].apply(lambda x: '↑' if x > 0.005 else ('↓' if x < -0.005 else '='))
df_delta = df_delta.sort_values('delta', ascending=False)

print(f'{"Classe":<35} {"Base":>8} {"+KW":>8} {"Delta":>8}')
print('-' * 60)
for _, r in df_delta.iterrows():
    print(f'{r["classe"]:<35} {r["f1_base"]:>8.4f} {r["f1_comb"]:>8.4f} {r["delta"]:>+8.4f}  {r["esito"]}')

print()
print(f'Macro F1  Baseline : {f1_base:.4f}')
print(f'Macro F1  Emb+KW   : {f1_comb:.4f}')
print(f'Delta              : {f1_comb - f1_base:+.4f}')

---
## STEP 8 — Coppie di confusione

In [ ]:
COPPIE = [
    ('ciclo_passivo',  'ciclo_attivo'),
    ('ciclo_attivo',   'ciclo_passivo'),
    ('area_personale', 'ciclo_passivo'),
    ('ciclo_attivo',   'area_sanitaria'),
    ('area_sanitaria', 'area_sistemistica'),
    ('area_personale', 'sistema381'),
]

print(f'{"Coppia (reale → predetto)":<45} {"Base":>6} {"+KW":>6} {"Rid.":>8}')
print('-' * 70)
for reale, predetto in COPPIE:
    n_b = ((y_test == reale) & (y_pred_base == predetto)).sum()
    n_c = ((y_test == reale) & (y_pred_comb == predetto)).sum()
    rid = n_b - n_c
    flag = '✓' if rid > 5 else ('✗' if rid < -5 else '')
    print(f'{reale} → {predetto:<25} {n_b:>6} {n_c:>6} {rid:>+8}  {flag}')